# Exploring the single-cell / single-nucleus atlas

Interactive tour of **`data/processed/dmg_atlas_v1.h5ad`** — the release object of the cross-cohort
H3 K27-altered diffuse midline glioma atlas (311,618 cells x 18,865 genes, 57 samples, 52 patients, 6 studies).

| | |
|---|---|
| file | `data/processed/dmg_atlas_v1.h5ad` (1.56 GB, md5 `4a39f172e4db0a2b470de7b6ac7b81e3`) |
| `X` | raw UMI counts, float32, CSR (601.9 M non-zeros, 10.2 % dense) |
| `obsm` | `X_scvi` (30-d), `X_scanvi` (30-d), `X_scvi_malignant` (30-d), `X_umap` (2-d, computed on `X_scanvi`) |
| schema | CELLxGENE 7.1.0, 0 MUST failures |
| sibling objects | `atlas_core.h5ad` = internal working copy (keeps rollback state, do not deposit); `query_mapping.h5ad` = 223,722 projected query cells (Section 9) |

**Requirements.** scanpy >= 1.11, anndata, ipywidgets, and ~6 GB of free RAM (the counts matrix is 4.8 GB in memory).
Run with the `multiverse` environment:
`/Users/jubayer/.claude-science/conda/envs/multiverse/bin/python -m jupyterlab`

Every panel is drawn from the object itself — nothing here is read from the manuscript tables, so the notebook
doubles as a check that the deposited file carries the figures' underlying values.

In [ ]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd, scanpy as sc
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ipywidgets import interact, Dropdown, Text, FloatSlider, fixed

ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / "data" / "processed").exists())
H5 = ROOT / "data/processed/dmg_atlas_v1.h5ad"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})
print("repo:", ROOT, "\nrelease:", H5.name, f"{H5.stat().st_size/1e9:.2f} GB")

In [ ]:
t0 = time.time()
adata = sc.read_h5ad(H5)
print(adata)
print(f"\nloaded in {time.time()-t0:.0f} s | X = {adata.X.dtype}, nnz {adata.X.nnz:,} "
      f"({100*adata.X.nnz/np.prod(adata.shape):.1f} % dense) | counts are integers: "
      f"{np.array_equal(adata.X.data[:100000], np.round(adata.X.data[:100000]))}")

## 1. What is inside

187 `obs` columns, grouped by what produced them. The 74 `author_*` columns are the original submitters'
per-barcode metadata, carried through unmodified; everything else is this pipeline's.

In [ ]:
groups = {
    "sample / donor":   ["sample_id", "patient_id", "donor_id", "study", "assay_atlas", "chemistry", "site",
                          "age_years", "age_group", "sex_atlas", "h3_variant", "k27m_status", "treatment",
                          "tissue_timing", "label_source"],
    "QC (phase 2)":      [c for c in adata.obs if c.startswith(("qc_", "n_", "pct_", "doublet", "predicted_", "high_mito"))],
    "CNV (phase 3)":     [c for c in adata.obs if c.startswith(("cnv_", "arm_", "chr")) or c in
                          ("marker_call", "marker_type", "consensus_call", "consensus_celltype_coarse",
                           "consensus_call_eventtrust", "malignant_call", "malignant_call_eventtrust")],
    "markers / scores":  [c for c in adata.obs if c.startswith(("score_", "gene_dev_"))],
    "models (phase 4)":  ["lineage_label", "scanvi_label", "scanvi_maxprob", "leiden_scanvi_r1",
                          "scanvi_validation_patient"],
    "metaprograms (6)":  [c for c in adata.obs if c.startswith("mp_")] + ["malignant_selected"],
    "microenvironment":  [c for c in adata.obs if c.startswith("tme_")],
    "CELLxGENE schema":  [c for c in adata.obs if c.endswith("_ontology_term_id")] +
                         ["suspension_type", "tissue_type", "is_primary_data", "cell_type_atlas"],
    "author_* (as submitted)": [c for c in adata.obs if c.startswith("author_")],
}
for k, v in groups.items():
    print(f"{k:26s} {len(v):3d}  {', '.join(v[:6])}{' ...' if len(v) > 6 else ''}")
print("\nunlisted:", sorted(set(adata.obs.columns) - {c for v in groups.values() for c in v}))
print("\nuns['release']:"); [print("  ", k, "=", v) for k, v in adata.uns["release"].items()]

In [ ]:
coh = (adata.obs.groupby("study", observed=True)
       .agg(cells=("sample_id", "size"), samples=("sample_id", "nunique"),
            patients=("patient_id", "nunique"),
            assays=("assay_atlas", lambda s: "+".join(sorted(set(s.astype(str))))),
            sites=("site", lambda s: "+".join(f"{k}:{v}" for k, v in s.value_counts().items() if v)),
            h3=("h3_variant", lambda s: "+".join(f"{k}:{s.eq(k).sum()}" for k in s.unique())))
       .sort_values("cells", ascending=False))
coh.index = coh.index.astype(str)
coh.loc["TOTAL"] = [adata.n_obs, adata.obs.sample_id.nunique(), adata.obs.patient_id.nunique(), "", "", ""]
display(coh)
print("nuclei fraction:", f"{adata.obs.suspension_type.eq('nucleus').mean():.3f}")

## 2. The integrated embedding

`X_umap` is computed on `X_scanvi` (k = 30 neighbours, UMAP seed 0) — not on `X_scvi`. scVI seed 1 was selected by a
pre-stated score (0.4 x batch mixing + 0.6 x biological conservation = 0.739, against Harmony 0.581 and PCA 0.126).

In [ ]:
UMAP = adata.obsm["X_umap"]

def umap_plot(key, ax=None, subset=None, s=0.6, cmap="viridis", legend=True, vmax_q=0.99, title=None):
    """Scatter X_umap coloured by any obs column (categorical -> legend, numeric -> colourbar)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 4.6))
    idx = np.ones(adata.n_obs, bool) if subset is None else np.asarray(subset)
    xy, v = UMAP[idx], adata.obs[key].values[idx]
    ax.scatter(*UMAP[~idx].T, s=s * 0.6, c="0.90", lw=0, rasterized=True) if (~idx).any() else None
    if str(adata.obs[key].dtype) in ("category", "object", "bool", "boolean"):
        v = pd.Series(v).astype(str).values
        cats = sorted(pd.unique(v))
        pal = (plt.cm.tab20.colors * 3)[:len(cats)] if len(cats) > 10 else plt.cm.tab10.colors[:len(cats)]
        for c, col in zip(cats, pal):
            m = v == c
            ax.scatter(*xy[m].T, s=s, color=col, lw=0, rasterized=True, label=f"{c} ({m.sum():,})")
        if legend:
            ax.legend(markerscale=12, fontsize=6, frameon=False, loc="center left",
                      bbox_to_anchor=(1.0, 0.5), ncol=1 + len(cats) // 16)
    else:
        v = v.astype(float)
        o = np.argsort(np.nan_to_num(v))
        sc_ = ax.scatter(*xy[o].T, s=s, c=v[o], cmap=cmap, lw=0, rasterized=True,
                         vmin=np.nanquantile(v, 0.01), vmax=np.nanquantile(v, vmax_q))
        plt.colorbar(sc_, ax=ax, shrink=0.6, label=key)
    ax.set(xticks=[], yticks=[], title=title or key)
    for sp in ax.spines.values():
        sp.set_visible(False)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
umap_plot("lineage_label", ax=axes[0], title="lineage (CNV + marker consensus)")
umap_plot("study", ax=axes[1], title="study of origin")
fig.tight_layout()

In [ ]:
COLOUR_BY = ["lineage_label", "cell_type_atlas", "scanvi_label", "scanvi_maxprob", "leiden_scanvi_r1",
             "study", "assay_atlas", "qc_assay_class", "site", "h3_variant", "age_group", "sex_atlas",
             "treatment", "tissue_timing", "consensus_call", "consensus_call_eventtrust", "cnv_call",
             "cnv_score", "cnv_confidence", "marker_call", "mp_family", "mp_top", "mp_max_z",
             "tme_compartment", "tme_annotation", "n_counts", "n_genes", "pct_mito", "pct_ribo",
             "doublet_score", "low_confidence_sample"]

@interact(key=Dropdown(options=COLOUR_BY, value="cell_type_atlas", description="colour by"),
          point_size=FloatSlider(0.6, min=0.2, max=3.0, step=0.2, description="point size"))
def _umap(key, point_size):
    umap_plot(key, s=point_size)
    plt.show()

## 3. Gene expression

`X` holds raw counts, so expression is log-normalised on the fly for the one gene requested (10^4 counts per cell,
`log1p`) — this avoids materialising a second 4.8 GB matrix. Genes are indexed by Ensembl ID; `var['gene_symbol']`
gives the symbol.

In [ ]:
sym2idx = pd.Series(np.arange(adata.n_vars), index=adata.var["gene_symbol"].astype(str).values)
sym2idx = sym2idx[~sym2idx.index.duplicated()]
size_factor = np.asarray(adata.X.sum(1)).ravel() / 1e4

def gene_vector(symbol):
    """log1p(CP10K) for one gene symbol."""
    if symbol not in sym2idx.index:
        raise KeyError(f"{symbol!r} not in the atlas ({adata.n_vars:,} genes)")
    x = np.asarray(adata.X[:, sym2idx[symbol]].todense()).ravel()
    return np.log1p(x / size_factor)

def gene_umap(symbol, ax=None, s=0.6):
    if ax is None:
        _, ax = plt.subplots(figsize=(5.2, 4.6))
    v = gene_vector(symbol)
    o = np.argsort(v)
    h = ax.scatter(*UMAP[o].T, s=s, c=v[o], cmap="magma_r", lw=0, rasterized=True,
                   vmax=max(np.quantile(v, 0.999), 1e-6))
    plt.colorbar(h, ax=ax, shrink=0.6, label="log1p(CP10K)")
    ax.set(xticks=[], yticks=[], title=f"{symbol}  ({100*(v>0).mean():.1f} % of cells detected)")
    for sp in ax.spines.values():
        sp.set_visible(False)
    return ax

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for g, ax in zip(["PDGFRA", "GFAP", "P2RY12"], axes):
    gene_umap(g, ax=ax)
fig.tight_layout()

In [ ]:
@interact(symbol=Text("MKI67", description="gene", continuous_update=False))
def _gene(symbol):
    symbol = symbol.strip().upper()
    if symbol in sym2idx.index:
        gene_umap(symbol); plt.show()
    else:
        hits = [g for g in sym2idx.index if g.startswith(symbol[:4])][:12]
        print(f"{symbol!r} not in the atlas. Similar: {hits}")

## 4. Malignant compartment and metaprograms

15 consensus metaprograms were derived by per-sample cNMF over the 167,395 consensus-malignant cells
(`malignant_selected`); 2 replicated under the pre-registered rule (>= 3 studies and both assays): MP1 cycling
(34 samples) and MP4 hypoxia (15 samples). MP2, MP12 and MP13 are technical (ribosomal, mitochondrial, dissociation
stress). `mp_top` is the argmax over the per-cell z-scores `mp_z_MP1 ... mp_z_MP15`.

In [ ]:
mal = adata.obs["malignant_selected"].values.astype(bool)
print(f"malignant_selected: {mal.sum():,} cells, {adata.obs.sample_id[mal].nunique()} samples")

MP_LABEL = {"MP1": "cycling", "MP2": "ribosomal*", "MP3": "OPC-like", "MP4": "hypoxia", "MP5": "OC-like",
            "MP6": "OPC-like", "MP7": "AC-like", "MP8": "MES-like", "MP9": "cilia / novel",
            "MP10": "AC-like", "MP11": "cycling", "MP12": "mitochondrial*", "MP13": "IEG stress*",
            "MP14": "AC-like", "MP15": "AC-like"}
mpz = [f"mp_z_MP{i}" for i in range(1, 16)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
umap_plot("mp_family", ax=axes[0], subset=mal, title="metaprogram family (malignant cells)")
heat = (adata.obs.loc[mal, mpz + ["mp_top"]].groupby("mp_top", observed=True).mean()
        .rename(columns=lambda c: f"{c[5:]} {MP_LABEL[c[5:]]}"))
im = axes[1].imshow(heat.values, cmap="RdBu_r", vmin=-2, vmax=4, aspect="auto")
axes[1].set(xticks=range(heat.shape[1]), yticks=range(heat.shape[0]), yticklabels=heat.index,
            ylabel="assigned mp_top", title="mean z-score per assigned program")
axes[1].set_xticklabels(heat.columns, rotation=90, fontsize=7)
plt.colorbar(im, ax=axes[1], shrink=0.7, label="mean z")
fig.tight_layout()

In [ ]:
@interact(program=Dropdown(options=[(f"{m[5:]} - {MP_LABEL[m[5:]]}", m) for m in mpz], value="mp_z_MP1",
                          description="program"),
          cells=Dropdown(options=["malignant only", "all cells"], value="malignant only"))
def _mp(program, cells):
    umap_plot(program, subset=mal if cells == "malignant only" else None, cmap="RdBu_r", vmax_q=0.995,
              title=f"{program[5:]} - {MP_LABEL[program[5:]]}")
    plt.show()

## 5. Copy-number evidence

Malignancy is a consensus of an inferCNV-style score and lineage markers. Pooled agreement with author labels is
0.8837 (sample median 0.898, IQR 0.793-0.978); cells whose call is backed by a recognised arm-level event
(`cnv_confidence == 'event_backed'`) reach precision 0.952 against 0.862 for score-only calls.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
ob = adata.obs
order = ["normal", "ambiguous", "malignant"]
axes[0].violinplot([ob.cnv_score[ob.consensus_call == c].dropna() for c in order], showmedians=True)
axes[0].set(xticks=[1, 2, 3], xticklabels=order, ylabel="cnv_score", title="CNV score by consensus call")

ct = pd.crosstab(ob.consensus_call, ob.cnv_confidence)
ct.plot.barh(stacked=True, ax=axes[1], width=0.7, color=["#2b6cb0", "#cbd5e0"])
axes[1].set(xlabel="cells", title="call support")
axes[1].legend(frameon=False, fontsize=7)

arms = ["chr1q_gain", "chr2_gain", "chr7_gain", "chr10q_loss", "chr13q_loss", "chr14q_loss", "chr17p_loss"]
frac = pd.DataFrame({c: ob.loc[ob.consensus_call == c, arms].eq("True").mean() for c in order})
frac.plot.bar(ax=axes[2], color=["#9ecae1", "#fdae6b", "#de2d26"], width=0.8)
axes[2].set(ylabel="fraction of cells with event", title="hallmark arm-level events")
axes[2].legend(frameon=False, fontsize=7); axes[2].tick_params(axis="x", rotation=60)
fig.tight_layout()
display(pd.crosstab(ob.consensus_call, ob.consensus_call_eventtrust, margins=True))

## 6. Microenvironment

21 microenvironment populations across four compartments, annotated from subclustered marker expression
(`tme_compartment` -> `tme_subcluster` -> `tme_annotation`).

In [ ]:
tme = adata.obs.tme_compartment.ne("none").values
fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), gridspec_kw={"width_ratios": [1.25, 1]})
umap_plot("tme_annotation", ax=axes[0], subset=tme, s=1.2, title="microenvironment annotation")
cnt = adata.obs.loc[tme].groupby(["tme_compartment", "tme_annotation"], observed=True).size().sort_values()
cnt.plot.barh(ax=axes[1], color="#4a5568")
axes[1].set(xscale="log", xlabel="cells (log scale)", ylabel="", title="population sizes")
axes[1].tick_params(labelsize=7)
fig.tight_layout()

## 7. Label-transfer confidence

The scANVI classifier reaches 0.9931 accuracy on its training cells and **0.9751 on 23,827 cells from 5 held-out
patients** (balanced accuracy 0.9166). T/NK precision is the weak point at 0.561.

In [ ]:
ho = adata.obs.scanvi_validation_patient.values.astype(bool)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1, 1.2]})
axes[0].hist([adata.obs.scanvi_maxprob[ho], adata.obs.scanvi_maxprob[~ho]], bins=40, density=True,
             label=[f"held-out patients (n={ho.sum():,})", f"training cells (n={(~ho).sum():,})"],
             color=["#dd6b20", "#2b6cb0"])
axes[0].set(xlabel="scanvi_maxprob", ylabel="density", title="posterior confidence")
axes[0].legend(frameon=False, fontsize=7)

cm = pd.crosstab(adata.obs.lineage_label[ho], adata.obs.scanvi_label[ho], normalize="index")
im = axes[1].imshow(cm.values, cmap="Blues", vmin=0, vmax=1)
axes[1].set(xticks=range(cm.shape[1]), yticks=range(cm.shape[0]), yticklabels=cm.index,
            ylabel="consensus label", xlabel="scANVI prediction", title="held-out patients: recall per class")
axes[1].set_xticklabels(cm.columns, rotation=60, ha="right")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if cm.values[i, j] > 0.01:
            axes[1].text(j, i, f"{cm.values[i, j]:.2f}", ha="center", va="center", fontsize=6,
                         color="white" if cm.values[i, j] > 0.5 else "black")
fig.tight_layout()

## 8. Composition across the cohort

Per-sample composition is what the manuscript's differential-abundance tests act on (Dirichlet-multinomial, patients
as the unit). The only contrast that survives multiple testing is neuron abundance between H3.1 and H3.3 samples
(q = 0.0040; site-adjusted 0.0169; pons-only 0.0495) — and neurons also track the nuclei-vs-cells contrast
(coefficient 3.40, q < 1e-9), so assay confounding is not excluded.

In [ ]:
def composition(group_by="h3_variant", label="lineage_label", ax=None):
    tab = (pd.crosstab(adata.obs[group_by], adata.obs[label], normalize="index")
           .reindex(columns=sorted(adata.obs[label].astype(str).unique())))
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4))
    tab.plot.bar(stacked=True, ax=ax, width=0.8, colormap="tab20", legend=False)
    ax.set(ylabel="fraction of cells", xlabel="", title=f"{label} composition by {group_by}", ylim=(0, 1))
    ax.legend(frameon=False, fontsize=6, loc="center left", bbox_to_anchor=(1, 0.5))
    ax.tick_params(axis="x", rotation=30)
    return tab

fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
composition("h3_variant", ax=axes[0]); composition("qc_assay_class", ax=axes[1])
fig.tight_layout()

@interact(group_by=Dropdown(options=["sample_id", "study", "site", "h3_variant", "assay_atlas", "qc_assay_class",
                                    "treatment", "tissue_timing", "age_group", "sex_atlas"], value="study"),
          label=Dropdown(options=["lineage_label", "cell_type_atlas", "scanvi_label", "mp_family",
                                  "tme_annotation", "consensus_call"], value="lineage_label"))
def _comp(group_by, label):
    fig, ax = plt.subplots(figsize=(max(6, 0.25 * adata.obs[group_by].nunique()), 4))
    composition(group_by, label, ax=ax); plt.show()

## 9. The projected query cohorts (separate file)

Five cohorts were mapped into the reference zero-shot and are deliberately **not** part of the atlas object, because
they never trained it: `data/processed/query_mapping.h5ad` (223,722 cells x 3,000 HVGs). Braun (fetal brain) and
Siletti (adult brain) are the negative controls — a malignant-label rate near zero is the expected result for
Siletti, and Braun's high rate reflects the genuine transcriptional similarity of OPC-like tumour cells to fetal
neural progenitors, not a failure.

Labels there are `predicted_lineage` (scANVI posterior) cross-checked against `knn_lineage` (reference-neighbour
vote); `classifier_knn_agree` flags where the two concur. This object is a **working** file and still carries
`*_pre_audit` / `*_fix1` columns from the audit — use the unsuffixed ones.

In [ ]:
q = sc.read_h5ad(ROOT / "data/processed/query_mapping.h5ad", backed="r")
qo = q.obs
per_sample = qo.groupby(["study", "sample_id"], observed=True).predicted_lineage.apply(
    lambda s: (s == "malignant").mean())
qs = (qo.groupby("study", observed=True)
      .agg(cells=("predicted_lineage", "size"), samples=("sample_id", "nunique"),
           pooled_malignant=("predicted_lineage", lambda s: (s == "malignant").mean()),
           median_maxprob=("scanvi_maxprob", "median"),
           knn_agreement=("classifier_knn_agree", lambda s: s.astype(str).eq("True").mean()))
      .join(per_sample.groupby("study", observed=True).median().rename("median_sample_malignant"))
      .sort_values("cells", ascending=False))
display(qs.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for st, g in qo.groupby("study", observed=True):
    axes[0].hist(g.scanvi_maxprob, bins=40, histtype="step", density=True, label=f"{st} (n={len(g):,})")
axes[0].set(xlabel="scanvi_maxprob", ylabel="density", title="projection confidence")
axes[0].legend(frameon=False, fontsize=7)
lin = pd.crosstab(qo.study, qo.predicted_lineage, normalize="index")
lin.plot.barh(stacked=True, ax=axes[1], colormap="tab10", width=0.8)
axes[1].set(xlabel="fraction of cells", ylabel="", title="transferred lineage composition")
axes[1].legend(frameon=False, fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
q.file.close()

## Where to go next

| what | where |
|---|---|
| spatial companion | `notebooks/spatial_atlas.ipynb` |
| per-cell malignancy calls (all 97 samples, pre-selection) | `data/processed/malignant_calls.csv` |
| metaprogram gene lists and replication | `results/tables/metaprogram_*.csv`, `docs/phases/phase6_notes.md` |
| QC trail, 10 studies 548,330 -> 455,766 cells | `results/qc/`, `docs/` |
| what was excluded and why | `docs/datasets/dmg_dataset_selection.csv` |
| release validation (114 pass / 0 MUST failures) | `docs/release/release_validation.md` |

**Caveats carried by this object.** 29.0 % of CNV-annotated cells are `ambiguous` and remain so; the H3 variant is
unreported for 25 of 57 samples (`h3_variant == 'unknown'`); 88.4 % of the atlas is nuclei, so assay class is
partially confounded with study and with neuron abundance; `is_primary_data` is False throughout, since every cell is
a reprocessing of a public deposit.